# 05. 메시지 삭제: RemoveMessage와 자동 삭제 전략

> 긴 대화는 컨텍스트 윈도우를 갉아먹어요. `RemoveMessage` 와 자동 삭제 전략으로 대화 이력을 안전하게 줄이는 기법을 정리해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `RemoveMessage` 수정자의 역할과 작동 원리를 설명할 수 있어요
2. `update_state()`를 사용해 외부에서 메시지를 수동으로 삭제할 수 있어요
3. `delete_messages` 노드를 그래프에 추가해 대화 이력을 자동으로 관리할 수 있어요
4. 메시지 개수 제한 전략으로 LLM 컨텍스트 윈도우와 토큰 비용을 관리할 수 있어요

## 사전 지식

- Part 02: `MessagesState`, `add_messages` reducer, `MemorySaver`로 체크포인팅
- Part 02: `get_state()`, `update_state()`로 그래프 상태 조회 및 수정
- Part 04: 체크포인트 복원력과 상태 이력 관리 (`04-Durable-Execution.ipynb`)

## 이전 노트북 연결

앞서 `04-Durable-Execution.ipynb`에서 체크포인트가 대화 이력을 누적 저장하는 방식을 배웠어요. 그런데 대화가 길어질수록 메시지가 계속 쌓이면 두 가지 문제가 생겨요: LLM 컨텍스트 윈도우 초과와 토큰 비용 증가. 이번 노트북에서는 이 문제를 해결하는 **메시지 삭제** 기법을 배울게요.


## RemoveMessage란?

`RemoveMessage`는 LangGraph에서 제공하는 특수 수정자(modifier)예요. 메시지 목록에서 특정 메시지를 **ID 기반으로 삭제**할 때 사용해요.

### 왜 메시지를 삭제해야 하나요?

대화가 계속되면 메시지가 쌓여요. 마치 책상 위에 서류가 쌓이는 것처럼요. 서류가 너무 많아지면 책상이 넘치죠(컨텍스트 윈도우 초과). 또한 LLM에게 모든 서류를 읽게 하면 시간도 오래 걸리고 비용도 많이 들어요(토큰 비용 증가). 그래서 오래된 서류는 정리하고 중요한 것만 남기는 거예요.

### 작동 원리

| 구성 요소 | 역할 | 특징 |
|----------|------|------|
| `RemoveMessage(id=...)` | 삭제 대상 지정 | 메시지 ID만 있으면 됨 |
| `add_messages` reducer | 실제 삭제 처리 | `RemoveMessage`를 인식하고 제거 |
| `MessagesState` | 기본 상태 타입 | `add_messages` reducer가 내장됨 |
| `update_state()` | 외부에서 상태 수정 | 그래프 외부에서 직접 호출 |

> 🔑 **핵심 개념**: `RemoveMessage`는 메시지를 직접 삭제하지 않아요. 대신 "이 ID의 메시지를 삭제하라"는 **신호**를 reducer에 보내요. `add_messages` reducer가 이 신호를 받아 실제로 상태에서 해당 메시지를 제거해요.

### 두 가지 삭제 방식

```mermaid
flowchart TB
    subgraph M ["수동 삭제 (Manual Delete)"]
        direction LR
        M1["get_state()로<br/>메시지 조회"]:::process
        M2["RemoveMessage<br/>(id=msg.id)"]:::process
        M3["update_state()<br/>호출"]:::process
        M4["get_state()로<br/>결과 확인"]:::output
        M1 --> M2 --> M3 --> M4
    end

    subgraph A ["자동 삭제 (Auto Delete)"]
        direction TB
        A1["agent 노드 실행"]:::input
        A2{"도구 호출 있음?"}:::process
        A3["action 노드 실행"]:::process
        A4["delete_messages 노드<br/>오래된 메시지 정리"]:::process
        A5["END"]:::output
        A1 --> A2
        A2 -->|예| A3
        A3 --> A1
        A2 -->|아니오| A4
        A4 --> A5
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

| 방식 | 사용 시점 | 장점 | 단점 |
|------|----------|------|------|
| **수동 삭제** | 민감 정보 제거, 디버깅 | 정밀한 타겟팅 | 매번 코드 실행 필요 |
| **자동 삭제** | 프로덕션 운영, 일상 관리 | 자동화, 일관성 | 중요 메시지 오삭제 위험 |


## 환경 설정


In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 불러와요
from dotenv import load_dotenv
load_dotenv()

# 환경 변수 로드 완료


In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택)
# ---------------------------------------------------
# LangSmith 프로젝트에서 메시지 삭제 흐름을 시각적으로 추적할 수 있어요
import os

# API 키가 있는 경우 아래 주석을 해제하세요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-DeleteMessages"

# 설정 완료


## 1. 기본 에이전트 그래프 구축

메시지 삭제를 테스트하기 위해 먼저 기본 에이전트 그래프를 만들어요. `MemorySaver` 체크포인터를 사용해 대화 이력을 축적하고, 거기에 `RemoveMessage`를 적용할 거예요.


In [ ]:
from typing import Literal

from langchain_core.tools import tool          # 도구 데코레이터
from langchain_core.messages import HumanMessage  # 사람의 메시지 타입
from langchain.chat_models import init_chat_model  # V1 모델 초기화
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode

model = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 필요한 모듈 임포트 및 기본 그래프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → {should_continue} → tool 또는 END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 대화 실행 - 메시지 누적 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 후속 대화 - 메시지가 계속 누적돼요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 현재 상태 확인 - 누적된 메시지 목록
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. 수동 메시지 삭제: update_state() + RemoveMessage

그래프 외부에서 특정 메시지를 직접 삭제하는 방법이에요. `update_state()`에 `RemoveMessage`를 전달하면 해당 ID의 메시지가 체크포인터에서 제거돼요.


In [ ]:
from langchain_core.messages import RemoveMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: RemoveMessage를 사용한 수동 삭제
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 삭제 결과 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.messages import RemoveMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 여러 메시지 동시 삭제
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 자동 메시지 삭제: delete_messages 노드

프로덕션 환경에서는 대화가 진행될수록 메시지가 계속 쌓여요. 이를 자동으로 관리하려면 그래프 안에 삭제 로직을 담은 노드를 추가해요.

핵심 전략: **에이전트가 최종 응답을 완료할 때마다** `delete_messages` 노드를 실행해 오래된 메시지를 정리해요.

### 자동 삭제 구현 핵심

```mermaid
flowchart LR
    A["agent 노드"] --> B{"should_continue"}
    B -->|"tool_calls 있음"| C["action 노드"]
    C --> A
    B -->|"tool_calls 없음<br/>(최종 응답 완료)"| D["delete_messages<br/>오래된 메시지 정리"]
    D --> E["END"]
    
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef delete fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    
    class A,C process
    class B decision
    class D delete
    class E process
```


In [ ]:
from typing import Literal
from langchain_core.tools import tool
from langchain_core.messages import RemoveMessage, HumanMessage
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode

model2 = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 자동 삭제 노드가 포함된 그래프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → {should_continue_v2} → action 또는 delete_messages → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 자동 삭제 동작 확인

실제로 대화를 여러 번 진행하면서 메시지가 어떻게 관리되는지 확인해볼게요. 매 대화 후 메시지 목록을 출력하면서 삭제 동작을 추적해요.

> 🔑 **핵심 개념**: 스트리밍 출력에서 메시지 수가 늘었다가 줄어드는 것을 볼 수 있어요. `delete_messages` 노드가 실행될 때 오래된 메시지가 상태에서 제거되는 순간을 직접 관찰할 수 있어요.


In [ ]:
from langchain_core.messages import HumanMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 첫 번째 대화 - 메시지 2개 누적 예상
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 두 번째 대화 - delete_messages 노드 작동 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 세 번째 대화 - 일관된 메시지 개수 유지 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 삭제 임계값 커스터마이징

메시지 개수 임계값을 변경하면 얼마나 많은 대화 이력을 유지할지 조절할 수 있어요. 더 많은 이력을 유지할수록 LLM 응답의 맥락이 풍부해지지만 토큰 비용이 늘어요.


In [ ]:
from langchain_core.messages import RemoveMessage, HumanMessage
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from typing import Literal

model3 = init_chat_model("openai:gpt-4o-mini")

# ============================================================
# TODO: 다른 임계값으로 자동 삭제 실험해보기
# 힌트: KEEP_LAST_N 숫자를 바꿔보세요
#       예: 최근 5개 유지 → KEEP_LAST_N = 5
#           최근 2개 유지 → KEEP_LAST_N = 2
# 예상 결과: 임계값이 크면 더 많은 대화 맥락이 유지돼요
# ============================================================


## 6. RemoveMessage의 고급 활용: 조건부 삭제

단순히 오래된 메시지를 삭제하는 것 외에도, 메시지의 **타입**이나 **내용**에 따라 선택적으로 삭제할 수 있어요.


In [ ]:
from langchain_core.messages import RemoveMessage, ToolMessage, HumanMessage, AIMessage

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 타입별 선택적 삭제: ToolMessage 필터링
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from collections import Counter

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 타입별 통계 확인 헬퍼
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`RemoveMessage`**: 메시지 ID를 지정해 상태에서 메시지를 제거하는 수정자예요. `add_messages` reducer가 이 신호를 처리해요. `langchain_core.messages`에서 임포트해요.
- **수동 삭제**: `update_state()` + `RemoveMessage(id=messages[0].id)` 패턴으로 외부에서 특정 메시지를 정밀하게 삭제해요.
- **자동 삭제**: `delete_messages` 노드를 그래프에 추가하고, `should_continue`에서 최종 응답 후 이 노드로 라우팅해요.
- **임계값 조절**: `messages[:-N]`에서 N을 바꿔 유지할 메시지 개수를 조절해요. 클수록 더 많은 맥락을 유지하지만 토큰 비용이 늘어요.
- **타입별 삭제**: `isinstance(m, ToolMessage)`처럼 메시지 타입을 확인해 선택적으로 삭제할 수 있어요.


## 다음 노트북 예고

다음 `06-Conversation-Summary.ipynb`에서는 **대화 요약 패턴**을 배워요. 메시지를 단순 삭제하면 맥락 손실이 생기는 문제를, LLM으로 오래된 대화를 **증분 요약(incremental summary)**해서 정보를 보존하면서도 토큰을 절약하는 방식으로 해결해요. 삭제와 요약을 조합하면 무한 대화도 실용적인 비용으로 운영할 수 있어요.
